# Despacho de Recursos de Emergencia — Simulación (Camino A)

**Tarea N°3 · ICS2123 Modelos Estocásticos · PUC**

Modelo de simulación de eventos discretos (cola de pérdida, sin espera) para el despacho de
unidades de emergencia en la Región Metropolitana, a partir de datos reales (SIGEB).

---

## Índice de secciones

1. **Datos** — carga, limpieza y derivación de columnas de apoyo.
2. **Modelo** — parámetros empíricos (tasa horaria, unidades por emergencia, tiempo de servicio).
3. **Motor** — calendario de eventos con `heapq`; modos agregado y por comuna.
4. **Verificación** — tests determinísticos, saturación, conservación y validación de volumen/perfil.
5. **Experimentos** — 3 escenarios × 2 resoluciones, 30 réplicas, IC 95%.
6. **Sensibilidad** — barridos de tiempo de servicio, flota y demanda.
7. **Export** — CSV, figuras y artefactos de resultados.

In [1]:
# Imports base del proyecto
import heapq
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('numpy', np.__version__)
print('pandas', pd.__version__)

numpy 2.5.0
pandas 3.0.3


## 1. Datos

Carga del dataset SIGEB, limpieza (filas con `Comuna`/`Material` nulos) y derivación de
columnas de apoyo (`hora`, `n_unidades`, `zona`). Salida: `datos_procesados.csv`.

In [2]:
# 2.1 Carga del dataset (ruta robusta con pathlib: funciona desde el notebook o la raíz)
def encontrar_xlsx(nombre='SIGEB_Estadisticas.xlsx'):
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'base' / nombre, base / 'tarea-3' / 'base' / nombre):
            if cand.exists():
                return cand
    raise FileNotFoundError(f'No se encontró {nombre} en base/ desde {Path.cwd()}')

RUTA_XLSX = encontrar_xlsx()
DIR_SALIDA = RUTA_XLSX.parent.parent / 'despacho_recursos'
print('Dataset:', RUTA_XLSX)

df = pd.read_excel(RUTA_XLSX, sheet_name='Sheet1')
df['Fecha'] = pd.to_datetime(df['Fecha'])
print('Filas cargadas:', len(df))

Dataset: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\base\SIGEB_Estadisticas.xlsx


Filas cargadas: 10082


In [3]:
# 2.2 Limpieza: quitar filas con Comuna o Material nulos; normalizar Comuna
n0 = len(df)
n_comuna_nula = int(df['Comuna'].isna().sum())
n_material_nulo = int(df['Material'].isna().sum())
mask_nulos = df['Comuna'].isna() | df['Material'].isna()
print(f'Comuna nula: {n_comuna_nula} | Material nulo: {n_material_nulo} | eliminadas (unión): {int(mask_nulos.sum())}')
df = df.loc[~mask_nulos].copy()
df['Comuna'] = df['Comuna'].str.strip().str.replace(r'\s+', ' ', regex=True)
print(f'Filas: {n0} -> {len(df)}')

Comuna nula: 3 | Material nulo: 1 | eliminadas (unión): 3
Filas: 10082 -> 10079


In [4]:
# 2.3 Columnas de apoyo: hora (0-23), n_unidades (conteo en Material), zona (9 desagregadas + Resto RM)
ZONAS_DESAGREGADAS = ['Santiago', 'Las Condes', 'Providencia', 'Estación Central', 'Renca',
                      'Independencia', 'Vitacura', 'Lo Barnechea', 'Recoleta']

df['hora'] = df['Fecha'].dt.hour
df['n_unidades'] = df['Material'].str.split(',').apply(lambda ts: sum(1 for t in ts if t.strip()))
df['zona'] = df['Comuna'].where(df['Comuna'].isin(ZONAS_DESAGREGADAS), 'Resto RM')

print('hora:', df['hora'].min(), '-', df['hora'].max())
print('n_unidades media:', round(df['n_unidades'].mean(), 4))
print('Distribución por zona:')
print(df['zona'].value_counts())

hora: 0 - 23
n_unidades media: 2.1451
Distribución por zona:
zona
Santiago            3412
Las Condes          1508
Providencia         1088
Estación Central    1049
Renca                730
Independencia        521
Vitacura             477
Lo Barnechea         472
Recoleta             443
Resto RM             379
Name: count, dtype: int64


In [5]:
# 2.4 Exportar dataset procesado (UTF-8)
RUTA_CSV = DIR_SALIDA / 'datos_procesados.csv'
df.to_csv(RUTA_CSV, index=False, encoding='utf-8')
print('Guardado:', RUTA_CSV)
print('Filas:', len(df), '| Columnas:', list(df.columns))

Guardado: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\despacho_recursos\datos_procesados.csv
Filas: 10079 | Columnas: ['Correlativo', 'Fecha', 'Clave', 'Calle', 'Esquina', 'Comuna', 'Material', 'hora', 'n_unidades', 'zona']
